# Cleaning My Scraped TikTok Data

### Imports

In [171]:
import json
import pandas as pd
from pathlib import Path

In [172]:
folder = Path(
    "/Users/annaclairebreuss-burgess/Library/Mobile Documents/"
    "com~apple~CloudDocs/AC Work/Navigator/"
    "navigator-travel-trends-intelligence/data/tiktok/raw"
)

destinations = [
    "lisbon",
    "marrakech",
    "hanoi",
    "reykjavik"
]

data = {}

for destination in destinations:
    data[destination] = []

    for file_number in [1, 2]:
        file_path = folder / f"{destination}{file_number}.json"

        with open(file_path, "r", encoding="utf-8") as f:
            data[destination].extend(json.load(f))


### Normalising and cleaning data

To combine df: 

combined_df = pd.concat(
    df.values(),
    ignore_index=True
)

To make hashtags not list - do before turning into a csv file

lisbon_df["hashtags"] = lisbon_df["hashtags"].apply(
    lambda x: ", ".join(x) if isinstance(x, list) else x
)

In [173]:
columns = [
    "url",
    "post_id",
    "description",
    "create_time",
    "digg_count",
    "share_count",
    "collect_count",
    "comment_count",
    "play_count",
    "video_duration",
    "hashtags",
    "profile_username",
    "profile_biography",
    "profile_followers",
    "is_verified",
    "discovery_input.search_keyword",
    "offical_item",
    "original_item",
    "music.authorname",
    "music.original",
    "music.title",
    "region",
    "timestamp"
]

date_columns = [
    "create_time",
    "timestamp"
]

numeric_columns = [
    "digg_count",
    "share_count",
    "collect_count",
    "comment_count",
    "play_count",
    "video_duration",
    "profile_followers"
]

df = {}

for destination in destinations:
    df[destination] = pd.json_normalize(
        data[destination]
    )

    df[destination] = df[destination][columns]

    df[destination]["destination"] = destination

    for column in date_columns:
        df[destination][column] = pd.to_datetime(
            df[destination][column], 
            errors="coerce"
        )

    for column in numeric_columns:
        df[destination][column] = pd.to_numeric(
            df[destination][column], 
            errors="coerce"
        )
    
    print(destination, df[destination].shape)

    df[destination] = df[destination].drop_duplicates(
        subset="post_id"
    )

for destination in destinations:
    print(destination, df[destination].shape)

# print("reykjavik", df[destination].dtypes)
# print("reykjavik", df[destination].head())


lisbon (2098, 24)
marrakech (2103, 24)
hanoi (2123, 24)
reykjavik (2110, 24)
lisbon (875, 24)
marrakech (853, 24)
hanoi (1097, 24)
reykjavik (901, 24)


## Cleaning data - looking at Na values and cleaning hashtags and descriptions

In [174]:
for destination in destinations:
    print(destination)

    print("Na values:")
    print(df[destination].isna().sum())

    print("Number of unique:")
    print("tikToks:", df[destination]["post_id"].nunique())
    print("creators:", df[destination]["profile_username"].nunique())

    df[destination]["hashtags"] = df[destination]["hashtags"].apply(
        lambda x: [hashtag.lower().strip() for hashtag in x]
        if isinstance(x, list)
        else []
    )

    # cleaning description by making everything lowercase and removing #
    df[destination]["description_clean"] = (
    df[destination]["description"]
        .fillna("")
        .str.lower()
        .str.replace(r"#", "", regex=True)
        .str.replace(r"\n", " ", regex=True)
        .str.strip()
    )

    # print(df[destination]["description_clean"].head())



lisbon
Na values:
url                                 0
post_id                             0
description                         3
create_time                         0
digg_count                          0
share_count                        57
collect_count                       0
comment_count                       0
play_count                          0
video_duration                    134
hashtags                           52
profile_username                    0
profile_biography                  38
profile_followers                   0
is_verified                         0
discovery_input.search_keyword      0
offical_item                        0
original_item                       0
music.authorname                    0
music.original                      0
music.title                         0
region                              0
timestamp                           0
destination                         0
dtype: int64
Number of unique:
tikToks: 875
creators: 702
marrakech
Na

Combining dataframes

In [175]:
# combined_df = pd.concat(
#     df.values(),
#     ignore_index=True
# )

# print(combined_df.shape)

# print(destination, combined_df["destination"].value_counts())

# combined_df.head()

### Adding to a csv files

In [176]:
from pathlib import Path

output_folder = Path(
    "/Users/annaclairebreuss-burgess/Library/Mobile Documents/"
    "com~apple~CloudDocs/AC Work/Navigator/"
    "navigator-travel-trends-intelligence/data/tiktok/processed"
)

output_folder.mkdir(parents=True, exist_ok=True)

for destination, destination_df in df.items():
    destination_df.to_csv(
        output_folder / f"{destination}_clean.csv",
        index=False
    )